In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import keras_tuner as kt

import copy
import random
import math

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split

from datetime import datetime

In [2]:
dbConnectionString = "sqlite:///baseball_info.db"
engine = create_engine(dbConnectionString)

In [106]:
timeSeriesHittingQuery = "SELECT SeasonStatsHitting.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2015 and 2025 and season != 2020 ORDER BY SeasonStatsHitting.playerId,season"

df = pd.read_sql(timeSeriesHittingQuery, engine)

In [107]:
def calculate_age_from_timestamps(birth_timestamp, current_timestamp):
    birth_date   = datetime.fromtimestamp(birth_timestamp)
    current_date = datetime.fromtimestamp(current_timestamp)

    age = current_date.year - birth_date.year

    # Adjust age if the birthday hasn't occurred yet in the current year
    if (current_date.month, current_date.day) < (birth_date.month, birth_date.day):
        age -= 1
    
    return age

In [108]:
twentyFifteenDateString     = "2015-04-05 00:00:00"
twentySixteenDateString     = "2016-04-03 00:00:00"
twentySeventeenDateString   = "2017-04-02 00:00:00"
twentyEighteenDateString    = "2018-03-29 00:00:00"
twentyNineteenDateString    = "2019-03-20 00:00:00"
twentyTwentyOneDateString   = "2021-04-01 00:00:00"
twentyTwentyTwoDateString   = "2022-04-07 00:00:00"
twentyTwentyThreeDateString = "2023-03-30 00:00:00"
twentyTwentyFourDateString  = "2024-03-20 00:00:00"
twentyTwentyFiveDateString  = "2025-03-18 00:00:00"

# Parse the string into a datetime object
twentyFifteenDateString     = datetime.strptime(twentyFifteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySixteenDateString     = datetime.strptime(twentySixteenDateString     , "%Y-%m-%d %H:%M:%S")
twentySeventeenDateString   = datetime.strptime(twentySeventeenDateString   , "%Y-%m-%d %H:%M:%S")
twentyEighteenDateString    = datetime.strptime(twentyEighteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyNineteenDateString    = datetime.strptime(twentyNineteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyTwentyOneDateString   = datetime.strptime(twentyTwentyOneDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyTwoDateString   = datetime.strptime(twentyTwentyTwoDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyThreeDateString = datetime.strptime(twentyTwentyThreeDateString , "%Y-%m-%d %H:%M:%S")
twentyTwentyFourDateString  = datetime.strptime(twentyTwentyFourDateString  , "%Y-%m-%d %H:%M:%S")
twentyTwentyFiveDateString  = datetime.strptime(twentyTwentyFiveDateString  , "%Y-%m-%d %H:%M:%S")

# Convert to Unix timestamp
twentyFifteenTimeStamp     = int(twentyFifteenDateString     .timestamp())
twentySixteenTimeStamp     = int(twentySixteenDateString     .timestamp())
twentySeventeenTimeStamp   = int(twentySeventeenDateString   .timestamp())
twentyEighteenTimeStamp    = int(twentyEighteenDateString    .timestamp())
twentyNineteenTimeStamp    = int(twentyNineteenDateString    .timestamp())
twentyTwentyOneTimeStamp   = int(twentyTwentyOneDateString   .timestamp())
twentyTwentyTwoTimeStamp   = int(twentyTwentyTwoDateString   .timestamp())
twentyTwentyThreeTimeStamp = int(twentyTwentyThreeDateString .timestamp())
twentyTwentyFourTimeStamp  = int(twentyTwentyFourDateString  .timestamp())
twentyTwentyFiveTimeStamp  = int(twentyTwentyFiveDateString  .timestamp())

seasonToStartTimeStamp = {}

seasonToStartTimeStamp[2015] = twentyFifteenTimeStamp   
seasonToStartTimeStamp[2016] = twentySixteenTimeStamp   
seasonToStartTimeStamp[2017] = twentySeventeenTimeStamp   
seasonToStartTimeStamp[2018] = twentyEighteenTimeStamp   
seasonToStartTimeStamp[2019] = twentyNineteenTimeStamp   
seasonToStartTimeStamp[2021] = twentyTwentyOneTimeStamp  
seasonToStartTimeStamp[2022] = twentyTwentyTwoTimeStamp  
seasonToStartTimeStamp[2023] = twentyTwentyThreeTimeStamp
seasonToStartTimeStamp[2024] = twentyTwentyFourTimeStamp 
seasonToStartTimeStamp[2025] = twentyTwentyFiveTimeStamp

In [109]:
#drop players that cannot create a valid window (ie have not played multiple season)
hasMultipleSeasons = df["playerId"].value_counts() > 1
df = df[df["playerId"].isin(hasMultipleSeasons[hasMultipleSeasons].index)]

#drop players with no plater appearances
df.drop(df[df['plateAppearances'] == 0].index, inplace=True)

# columnsToCheck = ['runs', 'homeRuns', 'rbis', 'stolenBases', 'onBasePercentage']

# zeroRowsMask = (df[columnsToCheck] != 0).any(axis=1)

# df = df[zeroRowsMask]

#get player ages
df['dob'] = df.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

logColumns     = df.columns[3:]
df[logColumns] = np.log(df[logColumns] + 1)

numericDf     = df.select_dtypes(include=np.number)
categoricalDf = df.select_dtypes(exclude=np.number)

mean = numericDf.mean()
std  = numericDf.std()

dfNormalized = (numericDf - mean) / std

dfNormalized = pd.concat([categoricalDf, dfNormalized], axis=1)

In [110]:
def getWindowedFeaturesAndLabels(frame, maxWindowSize):
    frameArray = frame.values.tolist()

    inputs = []
    labels = []
    
    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[5:6] + labelTest[10:12] + labelTest[18:19] + labelTest[23:24]

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [0] * 34
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    inputs.append(copy.deepcopy(windowArray))
                    labels.append(copy.deepcopy(relevantLabels))

                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    return inputs,labels

In [111]:
fullFeatures,fullLabels = getWindowedFeaturesAndLabels(dfNormalized, 4)

In [112]:
def getNaiveLoss(frame, maxWindowSize):
    meanSquaredError = 0.0
    labelCount       = 0.0
    
    frameArray = frame.values.tolist()

    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[5:6] + labelTest[10:12] + labelTest[18:19] + labelTest[23:24]

                    labelRuns        = relevantLabels[0]
                    labelHomeRuns    = relevantLabels[1]
                    labelRbis        = relevantLabels[2]
                    labelStolenBases = relevantLabels[3]
                    labelsObp        = relevantLabels[4]

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [0] * 33
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    lastWindow = windowArray[0]

                    relevantWindowLabels = lastWindow[5:6] + lastWindow[10:12] + lastWindow[18:19] + lastWindow[25:26]

                    naiveRuns        = relevantWindowLabels[0]
                    naiveHomeRuns    = relevantWindowLabels[1]
                    naiveRbis        = relevantWindowLabels[2]
                    naiveStolenBases = relevantWindowLabels[3]
                    naivesObp        = relevantWindowLabels[4]

                    meanSquaredError += pow(naiveRuns        - labelRuns       , 2)
                    meanSquaredError += pow(naiveHomeRuns    - labelHomeRuns   , 2)
                    meanSquaredError += pow(naiveRbis        - labelRbis       , 2)
                    meanSquaredError += pow(naiveStolenBases - labelStolenBases, 2)
                    meanSquaredError += pow(naivesObp        - labelsObp       , 2)

                    labelCount += 1.0
                    
                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    meanSquaredError = meanSquaredError / labelCount

    return meanSquaredError

In [113]:
naiveMse = getNaiveLoss(dfNormalized, 4)

naiveMse

6.995917553848452

In [114]:
fullFeaturesArray = np.array(fullFeatures)
fullLabelsArray   = np.array(fullLabels)

In [25]:
runBins              = np.linspace(min(fullLabelsArray[:,0]), max(fullLabelsArray[:,0]), num=4)
homeRunsBins         = np.linspace(min(fullLabelsArray[:,1]), max(fullLabelsArray[:,1]), num=4)
rbiBins              = np.linspace(min(fullLabelsArray[:,2]), max(fullLabelsArray[:,2]), num=4)
stolenBaseBins       = np.linspace(min(fullLabelsArray[:,3]), max(fullLabelsArray[:,3]), num=4)
onBasePercentageBins = np.linspace(min(fullLabelsArray[:,4]), max(fullLabelsArray[:,4]), num=4)

In [26]:
runsBinned             = np.digitize(fullLabelsArray[:,0], runBins, right=True)
homeRunsBinned         = np.digitize(fullLabelsArray[:,1], homeRunsBins, right=True)
rbisBinned             = np.digitize(fullLabelsArray[:,2], rbiBins, right=True)
stolenBasesBinned      = np.digitize(fullLabelsArray[:,3], stolenBaseBins, right=True)
onBasePercentageBinned = np.digitize(fullLabelsArray[:,4], onBasePercentageBins, right=True)

In [27]:
fullLabelsArrayBinned = np.column_stack((runsBinned, homeRunsBinned, rbisBinned, stolenBasesBinned, onBasePercentageBinned))

In [28]:
binnedDf = pd.DataFrame({
    'runs': runsBinned,
    'homeRuns': homeRunsBinned,
    'rbis': rbisBinned,
    'stolenBases': stolenBasesBinned,
    'onBasePercentage': onBasePercentageBinned
})

uniqueRowsIndexes = binnedDf.drop_duplicates(keep=False).index

uniqueRowsIndexesArray = np.array(uniqueRowsIndexes)
fullLabelsArrayBinnedFiltered = np.delete(fullLabelsArrayBinned, uniqueRowsIndexesArray, axis=0)
fullFeaturesArrayFiltered = np.delete(fullFeaturesArray, uniqueRowsIndexesArray, axis=0)
fullLabelsArrayFiltered = np.delete(fullLabelsArray, uniqueRowsIndexesArray, axis=0)

In [29]:
trainFeatures,valTestFeatures,trainLabels,valTestLables = train_test_split(fullFeaturesArrayFiltered, fullLabelsArrayFiltered, stratify=fullLabelsArrayBinnedFiltered, test_size=0.2)

valFeatures,testFeatures,valLabels,testLabels = train_test_split(valTestFeatures, valTestLables, test_size=0.5)

In [30]:
trainFeatures.shape

(3020, 3, 32)

In [38]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def getModelForTuner(hp):
    hp_learning_rate     = hp.Choice('learning_rate'    , values=[1e-2, 1e-3, 1e-4])
    hp_weight_decay      = hp.Choice('weight_decay'     , values=[1e-3,1e-4,1e-5])   

    num_rnn_layers = hp.Int("num_rnn_layers", min_value=1, max_value=3) # Example: 1 to 3 RNN layers
    
    model = tf.keras.Sequential()
    model.add(layers.Masking(mask_value=0., input_shape=(3, 28)))

    for i in range(num_rnn_layers):
        hp_units             = hp.Int   (f'units_{i}', min_value=5, max_value=30, step=6)
        hp_dropout           = hp.Choice(f'dropout_{i}'          , values=[0.2,0.3,0.4,0.5])     
        hp_recurrent_dropout = hp.Choice(f'recurrent_dropout_{i}', values=[0.2,0.3,0.4,0.5])  
        hp_regularizer       = hp.Choice(f'regularizer_{i}'      , values=[0.0001, 0.001, 0.01])  

        if i < num_rnn_layers - 1:
            model.add(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=True))
        else:
            model.add(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=False))
        
        model.add(layers.LayerNormalization())
        model.add(layers.Dense(units = 5))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate, weight_decay=hp_weight_decay), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [39]:
tuner = kt.Hyperband(getModelForTuner,
                     objective='val_loss',
                     max_epochs=100,
                     factor=3,
                     directory='tuner',
                     project_name='tuner_test')

C:\Users\graye\dev\python\fantasyBaseballPlayerData\player_projections\player_projections_env\Lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [40]:
tuner.search(trainFeatures, trainLabels, batch_size=32, epochs=50, validation_data=(valFeatures, valLabels))

Trial 254 Complete [00h 05m 31s]
val_accuracy: 0.5257353186607361

Best val_accuracy So Far: 0.6102941036224365
Total elapsed time: 02h 40m 44s


In [42]:
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

In [45]:
print(best_hps.get('num_rnn_layers'))
print(best_hps.get('learning_rate'))
print(best_hps.get('weight_decay'))
print(best_hps.get('units_0'))
print(best_hps.get('dropout_0'))
print(best_hps.get('recurrent_dropout_0'))
print(best_hps.get('regularizer_0'))


1
0.001
1e-05
23
0.2
0.5
0.001


In [31]:
trainFeatures.shape

(3020, 3, 32)

In [32]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def get_model():
    normalizeInput = layers.Normalization()
    normalizeInput.adapt(trainFeatures)

    model = tf.keras.Sequential([
        layers.Masking(mask_value=0., input_shape=(3, 32)),
        #normalizeInput,
        layers.LSTM(units=23, dropout=0.2, recurrent_dropout=0.5, kernel_regularizer=regularizers.l2(0.001), return_sequences=False),
        layers.LayerNormalization(),
        layers.Dense(units = 5)
    ])

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, weight_decay=1e-05), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [33]:
curModel = get_model()


curModel.summary()

C:\Users\graye\dev\python\fantasyBaseballPlayerData\player_projections\player_projections_env\Lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ masking (Masking)                    │ (None, 3, 32)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 23)                  │           5,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization                  │ (None, 23)                  │              46 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 5)                   │             120 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,318 (20.77 KB)

 Trainable params: 5,318 (20.77 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
curModel.fit(trainFeatures, trainLabels, batch_size=32, epochs=100, validation_data=(valFeatures, valLabels))

Epoch 1/100
95/95 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.2391 - loss: 0.4665 - r2_score: 0.0193 - val_accuracy: 0.3607 - val_loss: 0.2990 - val_r2_score: 0.4257
Epoch 2/100
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3096 - loss: 0.3162 - r2_score: 0.4049 - val_accuracy: 0.4483 - val_loss: 0.2560 - val_r2_score: 0.5184
Epoch 3/100
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3735 - loss: 0.2790 - r2_score: 0.4816 - val_accuracy: 0.4615 - val_loss: 0.2349 - val_r2_score: 0.5558
Epoch 4/100
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3980 - loss: 0.2543 - r2_score: 0.5263 - val_accuracy: 0.4589 - val_loss: 0.2210 - val_r2_score: 0.5793
Epoch 5/100
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4159 - loss: 0.2378 - r2_score: 0.5541 - val_accuracy: 0.4881 - val_loss: 0.2089 - val_r2_score: 0.6021
Epoch 6/100
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4381 - loss: 0.2280 - r2_score: 0.5679 - val_accuracy: 0.5172 - val_loss: 0.1994 - val_r

In [ ]:
accuracy: 0.5438 - loss: 0.1811 - r2_score: 0.6360 - val_accuracy: 0.5348 - val_loss: 0.1908 - val_r2_score: 0.5936

In [35]:
baselineResults = curModel.evaluate(testFeatures, testLabels)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5450 - loss: 0.1661 - r2_score: 0.6533 


In [36]:
for featureIndex in range(testFeatures.shape[2]):
   print(df.columns[featureIndex+2])
   shuffledTestFeatures = testFeatures.copy()

   testColumnElements = [shuffledTestFeatures[i][j][featureIndex] for i in range(len(shuffledTestFeatures)) for j in range(len(shuffledTestFeatures[i]))]

   random.shuffle(testColumnElements)
   
   k = 0
   
   for i in range(len(shuffledTestFeatures)):
       for j in range(len(shuffledTestFeatures[i])):
           shuffledTestFeatures[i][j][featureIndex] = testColumnElements[k]
   
           k += 1

   shuffledResults = curModel.evaluate(shuffledTestFeatures, testLabels)

   print(shuffledResults[1] / baselineResults[1])

dob
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4656 - loss: 0.2466 - r2_score: 0.4892 
0.7487975599942704
plateAppearances
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4656 - loss: 0.2397 - r2_score: 0.5082 
0.7778631337797188
atBats
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4577 - loss: 0.2448 - r2_score: 0.4901 
0.7501205222692491
runs
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4788 - loss: 0.2400 - r2_score: 0.5091 
0.7793208967002809
hits
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4524 - loss: 0.2402 - r2_score: 0.5053 
0.773490757374348
singles
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4577 - loss: 0.2405 - r2_score: 0.5064 
0.7751891998908822
doubles
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4630 - loss: 0.2457 - r2_score: 0.4924 
0.753683912329857
triples
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4709 - loss: 0.2425 - r2_score: 0.5009 
0.7667675124513829
homeRuns
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/s

In [96]:
#TestResults
playerFeaturesQuery = "SELECT Bios.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsHitting.playerId =\"660271\" ORDER BY SeasonStatsHitting.playerId,season"

playerDf = pd.read_sql(playerFeaturesQuery, engine)

In [97]:
playerDf['dob'] = playerDf.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

In [98]:
playerDf.transpose()

,0,1,2
playerId,660271,660271,660271
season,2023,2024,2025
dob,28,29,30
plateAppearances,594,728,725
atBats,497,636,611
runs,102,134,146
hits,151,197,172
singles,73,98,83
doubles,26,38,25
triples,8,7,9


In [99]:
logColumns           = playerDf.columns[3:]
playerDf[logColumns] = np.log(playerDf[logColumns] + 1)

numericPlayerDf     = playerDf.select_dtypes(include=np.number)
categoricalPlayerDf = playerDf.select_dtypes(exclude=np.number)

dfPlayerNormalized = (numericPlayerDf - mean) / std
dfPlayerNormalized = pd.concat([categoricalPlayerDf, dfPlayerNormalized], axis=1)

In [100]:
playerStats = []

playerFrameArray = dfPlayerNormalized.values.tolist()

for i in range(len(playerFrameArray)):
    playerStats.append(playerFrameArray[i][2:])

if len(playerStats) < 5:
    diff = 5 - len(playerStats)

    while diff > 0:
        playerStats.append([0] * 32)

        diff -= 1

In [101]:
playerWindow = np.array([playerStats])

playerPrediction = curModel.predict(playerWindow)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


In [102]:
indexToMean = {}

indexToMean[0] = mean.iloc[4]
indexToMean[1] = mean.iloc[9]
indexToMean[2] = mean.iloc[10]
indexToMean[3] = mean.iloc[17]
indexToMean[4] = mean.iloc[22]

indexToStd = {}

indexToStd[0] = std.iloc[4]
indexToStd[1] = std.iloc[9]
indexToStd[2] = std.iloc[10]
indexToStd[3] = std.iloc[17]
indexToStd[4] = std.iloc[22]

for i in range(len(playerPrediction[0])):
    prediction = playerPrediction[0][i]

    curMean = indexToMean[i]
    curStd  = indexToStd [i]

    prediction = prediction * curStd + curMean

    #value * std of column + mean of column
    prediction = math.exp(prediction) - 1

    print(prediction)

112.36366530767448
33.17721509755055
105.57358573392867
17.492114627551352
0.37397335108493657


In [79]:
mean[0]

C:\Users\graye\AppData\Local\Temp\ipykernel_6076\1567956375.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean[0]


np.float64(2021.3307573415766)

In [64]:
mean

np.float64(28.975270479134466)

In [ ]:
binnedDf = pd.DataFrame({
    'runs': runsBinned,
    'homeRuns': homeRunsBinned,
    'rbis': rbisBinned,
    'stolenBases': stolenBasesBinned,
    'onBasePercentage': onBasePercentageBinned
})

In [ ]:
https://en.wikipedia.org/wiki/Speed_Score#:~:text=Factor%201%20(Stolen%20base%20percentage,:%203B:%20SS:%20OF: